In [0]:
import pandas as pd
from pyspark.sql.functions import col, udf, sum 
from pyspark.sql.types import FloatType


In [0]:
pandas_transactions_file = '/dbfs/FileStore/tables/transactions/transactions.parquet'
spark_transactions_file = '/FileStore/tables/transactions/transactions.parquet'
pandas_products_file = '/dbfs/FileStore/tables/transactions/products.csv'
spark_products_file = '/FileStore/tables/transactions/products.csv'

disabling caching to get a correct estimation

In [0]:

spark.conf.set('spark.databricks.io.cache.enabled', False)


# Reading parquet file

With Pandas

In [0]:
%%timeit
pandas_df = pd.read_parquet(pandas_transactions_file)


In [0]:
pandas_df = pd.read_parquet(pandas_transactions_file)


With spark

In [0]:
%%timeit
spark_df = spark.read.options(header='true', inferSchema='true').parquet(spark_transactions_file)

In [0]:
spark_df = spark.read.options(header='true', inferSchema='true').parquet(spark_transactions_file)

For parquet file with 1M rows, spark seems to be 4 times more efficent to read data.

# Reading csv small static file

With Pandas 

In [0]:
%%timeit
pandas_products_df = pd.read_csv(pandas_products_file)

In [0]:
pandas_products_df = pd.read_csv(pandas_products_file)

With Spark

In [0]:
%%timeit 
spark_products_df = spark.read.options(header='true', inferSchema='true').csv(spark_products_file)

In [0]:
spark_products_df = spark.read.options(header='true', inferSchema='true').csv(spark_products_file)

Pandas is ~400x times better when dealing with small datasets

# Getting Stats on transaction files

With Pandas

In [0]:
%%timeit
pandas_df.describe()



In [0]:
%%timeit
pandas_products_df.describe()

With Spark

In [0]:
%%timeit
spark_df.describe()

In [0]:
%%timeit
spark_products_df.describe()


Spark is clearely more efficent to describe Big Datasets than pandas.   
Pandas is more efficent for smaller dataset 

# Testing Filters 

With Pandas 

In [0]:
%%timeit
pandas_df.loc[(pandas_df.region == 'Europe') & (pandas_df.price > 100)]

With Spark

In [0]:
%%timeit
spark_df.where((spark_df.region == 'Europe') & (spark_df.price > 100))

Spark is 10x faster on such a filter

# Transformations 

With Pandas

In [0]:
%%timeit 
pandas_df['revenue'] = pandas_df['price'] * pandas_df['quantity']

In [0]:
pandas_df['revenue'] = pandas_df['price'] * pandas_df['quantity']

With Spark

In [0]:
%%timeit 
spark_df.withColumn('revenue', col('price') * col('quantity'))

In [0]:
spark_df = spark_df.withColumn('revenue', col('price') * col('quantity'))


Multiplication is very simple operation that is optimised by Pandas and here pandas is very efficent in the calculation

In [0]:
@udf(FloatType())
def compliated_func(region, price, quantity, date):
    if str(date) < '2023-01-01':
        if 'Eur' in region:
            return price*quantity
        else:
            return 10*quantity
    elif str(date) < '2023-06-01':
        if 'Eur' in region:
            return 0
        return price*quantity
    return 1.030

def compliated_func_pandas(region, price, quantity, date):
    if str(date) < '2023-01-01':
        if 'Eur' in region:
            return price*quantity
        else:
            return 10*quantity
    elif str(date) < '2023-06-01':
        if 'Eur' in region:
            return 0
        return price*quantity
    return 1.030



Applying complicated func to `Pandas`

In [0]:
%%timeit 
pandas_df['new_column'] = [compliated_func_pandas(region, price, quantity, date) for region, price, quantity, date in zip(pandas_df['region'], pandas_df['price'], pandas_df['quantity'], pandas_df['transaction_date'])]


Applying complicated func to `Spark`

In [0]:
%%timeit
spark_df.withColumn('new_column', compliated_func(col('region'), col('price'), col('quantity'), col('transaction_date')))


Here we can see that Spark is 10x more efficent then Pandas 

# Aggregation

With Pandas

In [0]:
%%timeit 
pandas_df.groupby('region').sum('revenue')

With Spark

In [0]:
%%timeit
spark_df.groupBy('region').agg(sum('revenue').alias('total_revenue'))

Again Spark is ~ 5x more performant on aggregation

# Sorting Data

With Pandas

In [0]:
%%timeit 
pandas_df.sort_values(by = 'revenue', ascending=False)

With Spark

In [0]:
%%timeit
spark_df.sort(spark_df.revenue.desc())

Spark is ~ 30x more efficent 

# Join tables 

With Pandas

In [0]:
%%timeit
pd.merge(pandas_df, pandas_products_df, on='product_id')

In [0]:
merged_pandas_df = pd.merge(pandas_df, pandas_products_df, on='product_id')

With Spark

In [0]:
%%timeit
spark_df.join(spark_products_df, spark_df.product_id == spark_products_df.product_id)

In [0]:
spark_products_df = spark_products_df.withColumnRenamed("product_id", "product_id_right")
merged_spark_df = spark_df.join(spark_products_df, on=spark_df.product_id == spark_products_df.product_id_right)

Spark is ~30x more efficent 

# Writing files 

In [0]:
pandas_transactions_file = '/dbfs/FileStore/tables/transactions/transactions.parquet'
spark_transactions_file = '/FileStore/tables/transactions/transactions.parquet'
pandas_products_file = '/dbfs/FileStore/tables/transactions/products.csv'
spark_products_file = '/FileStore/tables/transactions/products.csv'

With Pandas

In [0]:
%%timeit 
merged_pandas_df.to_parquet('/dbfs/FileStore/tables/transactions/merged_pandas_df.parquet')

In [0]:
%%timeit
merged_spark_df.write.mode('append').parquet('/FileStore/tables/transactions/merged_spark_df.parquet')


Pandas is more efficent in writting files 